In [2]:
from scenario_access import load_scenario

data=load_scenario(32,"",False)

********************************************************
Loading Scenario32 dataset from 


C:\Users\Sedat Cimen\deepsense_positon_non_linear\scenario_access.py:92: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  scenario_df.interpolate(method='linear', axis=0, limit_direction='both',inplace=True)


In [3]:
data.head()

,unit2_spd_over_grnd_kmph,unit2_altitude,unit2_geo_sep,unit2_mode_fix_type,unit2_pdop,unit2_hdop,unit2_vdop,unit2_interpolated_position,unit2_lat,unit2_lon,unit1_beam_index
0,5.820,356.810,-27.749,3.0,1.09,0.58,0.92,0.0,33.424312,-111.935136,2
1,5.841,356.812,-27.749,3.0,1.09,0.58,0.92,1.0,33.424312,-111.935134,2
2,5.862,356.814,-27.749,3.0,1.09,0.58,0.92,0.0,33.424312,-111.935133,2
3,6.066,356.814,-27.749,3.0,1.09,0.58,0.92,0.0,33.424312,-111.935133,2
4,6.351,356.807,-27.749,3.0,1.09,0.58,0.92,0.0,33.424312,-111.935131,4


In [7]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

# --- GEOMETRİK HESAPLAMALAR ---

def get_distance(lat1, lon1, lat2, lon2):
    R = 6371000 # metre cinsinden dünya yarıçapı
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))

def get_bearing(lat1, lon1, lat2, lon2):
    dLon = np.radians(lon2 - lon1)
    y = np.sin(dLon) * np.cos(np.radians(lat2))
    x = np.cos(np.radians(lat1)) * np.sin(np.radians(lat2)) - \
        np.sin(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.cos(dLon)
    return np.degrees(np.arctan2(y, x))

# Sabit Baz İstasyonu 
u1_lat, u1_lon = 33.4251, -111.9352 

# Öznitelikler
data['dist_to_bs'] = get_distance(u1_lat, u1_lon, data['unit2_lat'], data['unit2_lon'])
data['bearing'] = get_bearing(u1_lat, u1_lon, data['unit2_lat'], data['unit2_lon'])
# Mevcut sütunların üzerine farkları (vektörleri) ekle
data['lat_diff'] = data['unit2_lat'].diff().fillna(0)
data['lon_diff'] = data['unit2_lon'].diff().fillna(0)

In [8]:
# Kullanılacak özellik listesi 
features = [
    'unit2_lat', 'unit2_lon', 'unit2_spd_over_grnd_kmph', 
    'unit2_altitude', 'unit2_pdop', 'dist_to_bs', 'bearing', 
    'lat_diff', 'lon_diff'
]

# Normalizasyon: Veriyi 0-1 arasına çekiyoruz
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(data[features])
y_raw = data['unit1_beam_index'].values

# Kayan Pencere (Windowing): Son 10 adımı paketle
def create_windows(X, y, window_size=10):
    X_win, y_win = [], []
    for i in range(len(X) - window_size):
        X_win.append(X[i : i + window_size])
        y_win.append(y[i + window_size])
    return np.array(X_win), np.array(y_win)

X_final, y_final = create_windows(X_scaled, y_raw, window_size=10)

# Dağıtımı yapıyoruz (%20 dürüst sınav seti)
X_train, X_test, y_train, y_test = train_test_split(X_final, y_final, test_size=0.2, random_state=42)

In [9]:
# 64 farklı hüzme 
num_classes = 64 

model = Sequential([
    LSTM(128, input_shape=(10, len(features)), return_sequences=True),
    BatchNormalization(),
    Dropout(0.3),
    LSTM(64),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dense(num_classes, activation='softmax')
])

model.compile(optimizer=Adam(learning_rate=0.001), 
              loss='sparse_categorical_crossentropy', 
              metrics=['accuracy'])

# --- FINE-TUNE ARAÇLARI 
# EarlyStopping: 
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# ReduceLROnPlateau: 
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=0.00001)

# EĞİTİM
history = model.fit(
    X_train, y_train,
    epochs=100, # 
    batch_size=32,
    validation_data=(X_test, y_test),
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

C:\Users\Sedat Cimen\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/100
81/81 ━━━━━━━━━━━━━━━━━━━━ 6s 19ms/step - accuracy: 0.1767 - loss: 3.1714 - val_accuracy: 0.1752 - val_loss: 3.1977 - learning_rate: 0.0010
Epoch 2/100
81/81 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.2543 - loss: 2.4961 - val_accuracy: 0.2109 - val_loss: 2.8753 - learning_rate: 0.0010
Epoch 3/100
81/81 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.2822 - loss: 2.3284 - val_accuracy: 0.2124 - val_loss: 2.5125 - learning_rate: 0.0010
Epoch 4/100
81/81 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.2903 - loss: 2.2827 - val_accuracy: 0.3504 - val_loss: 2.2607 - learning_rate: 0.0010
Epoch 5/100
81/81 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.3023 - loss: 2.1851 - val_accuracy: 0.3163 - val_loss: 2.2535 - learning_rate: 0.0010
Epoch 6/100
81/81 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.3124 - loss: 2.1396 - val_accuracy: 0.2992 - val_loss: 2.2423 - learning_rate: 0.0010
Epoch 7/100
81/81 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.3326 - loss: 2.0512 - 

In [10]:
import tensorflow as tf

# Modelin test seti üzerindeki olasılık tahminlerini alalım
y_pred_probs = model.predict(X_test)

# Top-1 Accuracy (Zaten bildiğimiz %42)
top1_acc = tf.keras.metrics.sparse_top_k_categorical_accuracy(y_test, y_pred_probs, k=1)
# Top-3 Accuracy
top3_acc = tf.keras.metrics.sparse_top_k_categorical_accuracy(y_test, y_pred_probs, k=3)
# Top-5 Accuracy (Genelde makalelerde bu da verilir)
top5_acc = tf.keras.metrics.sparse_top_k_categorical_accuracy(y_test, y_pred_probs, k=5)

print(f"Top-1 Doğruluk (Tam İsabet): %{np.mean(top1_acc)*100:.2f}")
print(f"Top-3 Doğruluk: %{np.mean(top3_acc)*100:.2f}")
print(f"Top-5 Doğruluk: %{np.mean(top5_acc)*100:.2f}")

21/21 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step
Top-1 Doğruluk (Tam İsabet): %42.95
Top-3 Doğruluk: %76.12
Top-5 Doğruluk: %87.44
